In [391]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"U:\data\HMC\HMC1\day8"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="Rat"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
theta_epochs = nel.EpochArray(
    state_dict["THETA"],
)
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")
n_pyr=len(E_spikes)
print(f"number of pyramidal cells: {n_pyr}")
print(f"nrem duration: {nrem_epochs.durations.sum():.2f} seconds")
print(f"rrem duration: {rem_epochs.durations.sum():.2f} seconds")
print(f"theta duration: {theta_epochs.durations.sum():.2f} seconds")

Folder already exists: U:\data\HMC\HMC1\day13\complex_system
number of cells: 105
number of pyramidal cells: 95
nrem duration: 11199.00 seconds
rrem duration: 1449.00 seconds
theta duration: 1101.00 seconds


In [334]:
import os, re, numpy as np, pandas as pd
import glob
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Coerce a MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved_error_bar_50ms.mat", f"*_{region}_D2_time_resolved_error_bar_50ms.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Load the CA1 d2 timeline
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day13_CA1_D2_time_resolved_error_bar_50ms.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.05, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 158, 'total_spikes': 6976785}
   State 0: NREM - requested points=3966, actual used=3966
     Time range: 814.0 to 23078.0
     D2 range: 0.0483 to 0.2280
     D2_ERR range: 0.0140 to 0.0352
   State 1: REM - requested points=414, actual used=414
     Time range: 3254.0 to 20113.0
     D2 range: 0.0178 to 0.1498
     D2_ERR range: 0.0080 to 0.0337
   State 2: WAKE - requested points=4590, actual used=4590
     Time range: 21.0 to 24536.0
     D2 range: 0.0117 to 0.2636
     D2_ERR range: 0.0088 to 0.0449

✅ Complete timeline created:
📊 Total data points: 8970
📊 Time range: 21.0 to 24536.0
📊 Duration: 24515.0 seconds
📊 State distribution:
state
WAKE    4590
NREM    3966
REM      414
Name: count, dtype: int64



In [337]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]

In [339]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Compute MUA firing rate with 0.04 second bins
time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

Computing MUA firing rate from spike times...


In [340]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# spike_times
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges → make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average of MUA (Hz)
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window; allow some tolerance near edges
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

# 4) Interpolate the smoothed MUA onto the D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)
mua_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe (Hz)
ca1_complete_df["MUA"] = mua_at_centers
ca1_complete_df["n_cell"] = n_cell

SWR and barrage

In [341]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs

try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    # Automatically detect the HSE filename based on basename
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    # brpath=os.path.join(basepath, "BARR_archive","Barrage_Files")
    # mat_path = os.path.join(brpath, f'{basename}.HSEn2.events.mat')

    # data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    # barrage = data['HSEn2']
    # barrage_timestamps= barrage.timestamps 
    # barrage_epochs=nel.EpochArray(barrage_timestamps)
    # barrage_epochs=nrem_epochs&barrage_epochs
    # Create barrage epochs
    # Find barrages in NREM epochs
    # #    ---- 1) Define paths ----
    # barrage_path = os.path.join(basepath, "Barrage_Files")

    # # Automatically detect the HSE filename based on basename
    # basename = os.path.basename(basepath)
    # matfile = f"{basename}.clean.HSE.mat"
    # fullfile = os.path.join(barrage_path, matfile)

    # # # ---- 2) Load .mat file ----

    # # # MATLAB variables available:
    # # #   timestamps : Nx2 array
    # mat = sio.loadmat(fullfile, squeeze_me=True, struct_as_record=False)
    # HSE = mat["HSE"]

    # timestamps = np.asarray(HSE.timestamps)
    # keep       = np.asarray(HSE.keep).squeeze().astype(int)   # indices into timestamps (1-based)
    # NREM       = np.asarray(HSE.NREM).squeeze().astype(int)   # indices into keep (1-based)

    # # # --- MATLAB: keep(NREM) ---
    # # # In MATLAB: idx_ts_1 = keep(NREM)
    # # # Step 1: convert NREM to 0-based to index keep
    # idx_keep_0 = NREM - 1              # 0-based indices into keep
    # idx_ts_1   = keep[idx_keep_0]      # still 1-based indices into timestamps

    # # # Step 2: convert to 0-based to index timestamps in Python
    # idx_ts_0   = idx_ts_1 - 1

    # # # --- MATLAB: timestamps(keep(NREM), :) ---
    # nrem_intervals = timestamps[idx_ts_0, :]

    # # # ---- Convert to nel.EpochArray ----
    # barrage_epoch = nel.EpochArray(nrem_intervals)
    barrage_start = barrages_in_nrem.data[:, 0]
        
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times

No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for U:\data\HMC\HMC1\day13


ISI entropy

In [385]:
# from pathlib import Path
# out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats")
# out_dir.mkdir(parents=True, exist_ok=True)

# fname = f"{animal_name}_{day}_{brain_region}.pkl"
# save_path = out_dir / fname

# ca1_complete_df.to_pickle(save_path)
# print(f"Saved to: {save_path}")

In [386]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from typing import Iterable

# # =========================
# # ISI entropy
# # =========================
# def isi_entropy(spike_times, n_bins=20):
#     """
#     Shannon entropy (bits) of log10(ISI) distribution.
#     Returns NaN if fewer than 2 ISIs (i.e., <3 spikes).
#     """
#     st = np.asarray(spike_times, float)
#     st = st[np.isfinite(st)]
#     st = np.unique(np.sort(st))
#     if st.size < 3:
#         return np.nan
#     isis = np.diff(st)
#     isis = isis[np.isfinite(isis) & (isis > 0)]
#     if isis.size < 2:
#         return np.nan
#     x = np.log10(isis)
#     hist, _ = np.histogram(x, bins=n_bins)
#     N = hist.sum()
#     if N <= 0:
#         return np.nan
#     p = hist.astype(float) / N
#     p = p[p > 0]
#     return float(-np.sum(p * np.log2(p)))  # bits

# # =========================
# # Utilities
# # =========================
# def _flatten_spikes(spike_times):
#     """Pool spike times across inputs and return sorted unique float seconds."""
#     if isinstance(spike_times, dict):
#         parts = [np.asarray(v, float).ravel() for v in spike_times.values()]
#         x = np.concatenate([p[np.isfinite(p)] for p in parts]) if parts else np.array([], float)
#     else:
#         arr = np.asarray(spike_times, dtype=object)
#         if arr.dtype == object:
#             parts = []
#             for it in arr:
#                 if it is None:
#                     continue
#                 v = np.asarray(it, float).ravel()
#                 v = v[np.isfinite(v)]
#                 if v.size:
#                     parts.append(v)
#             if parts:
#                 x = np.concatenate(parts)
#             else:
#                 try:
#                     x = np.asarray(spike_times, float).ravel()
#                 except (TypeError, ValueError):
#                     x = np.array([], float)
#         else:
#             x = arr.astype(float).ravel()
#     x = x[np.isfinite(x)]
#     return np.sort(np.unique(x))

# def _coerce_epochs(ep):
#     """Coerce many epoch formats to a sorted list[(start, stop)]. Accepts ep, or ep.data with ≥2 columns."""
#     # Try .data first (Epoch-like objects)
#     if hasattr(ep, "data"):
#         vals = getattr(ep, "data")
#         try:
#             if isinstance(vals, pd.DataFrame) and vals.shape[1] >= 2:
#                 a = vals.iloc[:, 0].astype(float).to_numpy()
#                 b = vals.iloc[:, 1].astype(float).to_numpy()
#                 out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#                 out.sort(key=lambda t: t[0]); return out
#             vals = np.asarray(vals)
#             if vals.ndim == 2 and vals.shape[1] >= 2:
#                 a = vals[:, 0].astype(float); b = vals[:, 1].astype(float)
#                 out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#                 out.sort(key=lambda t: t[0]); return out
#         except Exception:
#             pass

#     # DataFrame
#     if isinstance(ep, pd.DataFrame):
#         if {'start','stop'}.issubset(ep.columns):
#             a = ep['start'].astype(float).to_numpy()
#             b = ep['stop'].astype(float).to_numpy()
#         elif {'startTime','stopTime'}.issubset(ep.columns):
#             a = ep['startTime'].astype(float).to_numpy()
#             b = ep['stopTime'].astype(float).to_numpy()
#         else:
#             a = ep.iloc[:,0].astype(float).to_numpy()
#             b = ep.iloc[:,1].astype(float).to_numpy()
#         out = [(s, e) for s, e in zip(a, b) if np.isfinite(s) and np.isfinite(e) and e > s]
#         out.sort(key=lambda t: t[0]); return out

#     # Iterable of pairs
#     if isinstance(ep, Iterable):
#         out = []
#         for item in ep:
#             try:
#                 s, e = item
#                 s, e = float(s), float(e)
#                 if np.isfinite(s) and np.isfinite(e) and e > s:
#                     out.append((s, e))
#             except Exception:
#                 pass
#         out.sort(key=lambda t: t[0]); return out

#     return []

# def _count_events(event_epochs, a, b):
#     """Count events whose mid-point falls in [a,b)."""
#     c = 0
#     for s, e in event_epochs:
#         if not np.isfinite(s) or not np.isfinite(e) or e <= s:
#             continue
#         mid = 0.5 * (s + e)
#         if a <= mid < b:
#             c += 1
#     return c
# def _centers_within_long_epochs(centers_s, epochs_data, min_dur=60.0):
#     """
#     centers_s: 1D array of center times (sec)
#     epochs_data: (N,2) array-like [[start,end], ...] in seconds
#     """
#     ep = np.asarray(epochs_data, dtype=float)
#     if ep.size == 0:
#         return np.zeros(len(centers_s), dtype=bool)

#     # keep only epochs with duration >= min_dur
#     dur = ep[:, 1] - ep[:, 0]
#     ep = ep[dur >= float(min_dur)]
#     if ep.size == 0:
#         return np.zeros(len(centers_s), dtype=bool)

#     # assume epochs are sorted; if not, sort by start
#     order = np.argsort(ep[:, 0])
#     ep = ep[order]
#     starts, ends = ep[:, 0], ep[:, 1]

#     c = np.asarray(centers_s, dtype=float)

#     # membership via searchsorted (O(M log N))
#     idx = np.searchsorted(starts, c, side="right") - 1
#     ok = (idx >= 0) & (c < ends[np.clip(idx, 0, len(ends)-1)])
#     return ok
# # =========================
# # Core: ISI entropy per window + ripple/barrage rates
# # =========================
# def isi_entropy_windows_with_tiles(
#     spike_times,
#     centers_df,                 # must have columns: center_s, state
#     ripple_epochs,
#     barrage_epochs,
#     state_filter="NREM",
#     window=10.0,                # seconds (±15s)
#     min_spikes=3,               # require ≥3 spikes in window to form ≥2 ISIs
#     n_bins_isi=20               # histogram bins for log10(ISI)
# ):
#     st = _flatten_spikes(spike_times)
#     out_cols = ["center_s","state","isi_entropy","n_spikes","ripple_rate","barrage_rate"]
#     if st.size == 0:
#         return pd.DataFrame(columns=out_cols)

#     # centers (NREM only by default)
#     # centers (NREM only by default)
#     # --- with the extra epoch filter ---
#     # NREM_epochs.data is (n,2) start/end in seconds
#     long_nrem_data = nrem_epochs.data[(nrem_epochs.data[:, 1] - nrem_epochs.data[:, 0]) >= 120]
#     if "state" in centers_df.columns:
#         sel = centers_df["state"].astype(str).str.upper() == str(state_filter).upper()
#         base = centers_df.loc[sel & centers_df["center_s"].notna()].copy()
#     else:
#         base = centers_df.loc[centers_df["center_s"].notna()].copy()
#         base["state"] = state_filter

#     in_long_nrem = _centers_within_long_epochs(base["center_s"].to_numpy(float), long_nrem_data, min_dur=120)
#     cdf = base.loc[in_long_nrem].copy()

#     if centers.size == 0:
#         return pd.DataFrame(columns=out_cols)

#     rip = _coerce_epochs(ripple_epochs)
#     bar = _coerce_epochs(barrage_epochs)

#     half = window/2.0
#     rows = []
#     for i, (c, s) in enumerate(zip(
#         cdf["center_s"].to_numpy(float),
#         cdf["state"].astype(str).to_numpy()
#     )):
#         a, b = c - half, c + half
#         # spikes in window
#         wst = st[(st >= a) & (st < b)]
#         nsp = int(wst.size)

#         if nsp >= min_spikes:
#             H = isi_entropy(wst, n_bins=n_bins_isi)
#         else:
#             H = np.nan

#         dur = b - a
#         if dur > 0:
#             rr = _count_events(rip, a, b) / dur
#             br = _count_events(bar, a, b) / dur
#         else:
#             rr = np.nan; br = np.nan

#         # start with all original metadata from centers_df (current row index)
#         row_dict = cdf.iloc[i].to_dict()

#         # add new fields
#         row_dict.update({
#             "isi_entropy": H,
#             "n_spikes": nsp,
#             "ripple_rate": rr,
#             "barrage_rate": br
#         })

#         rows.append(row_dict)

#     out = pd.DataFrame(rows).sort_values("center_s").reset_index(drop=True)
#     print(out.head())
#     return out

# # =========================
# # Tile aggregation (6 tiles) and plotting
# # =========================

# # =========================
# # Example usage
# # =========================
# # Required inputs:
# #   - spike_times: array/list/dict of spike timestamps (sec)
# #   - ca1_complete_df: DataFrame with ['center_s','state']
# #   - ripple_epochs, barrage_epochs: epoch objects (with .data), DataFrames, or list[(start,stop)]
# #import numpy as np
# import pandas as pd

# def filter_df_by_epoch_names(df, epoch_df, names=("pre","post"),
#                             tcol="center_s", start_col="startTime", stop_col="stopTime", name_col="name"):
#     out = {}
#     t = df[tcol].to_numpy(float)

#     for nm in names:
#         iv = epoch_df.loc[epoch_df[name_col].eq(nm), [start_col, stop_col]].to_numpy(float)
#         if iv.size == 0:
#             out[nm] = df.iloc[0:0].copy()
#             continue

#         mask = np.zeros(len(df), dtype=bool)
#         for start, stop in iv:
#             mask |= (t >= start) & (t < stop)   # [start, stop)
#         out[nm] = df.loc[mask].copy()

#     return out

# def mask_times_in_intervals(t, intervals):
#     """
#     t: 1D times
#     intervals: (n,2) array [[start, stop], ...]
#     returns: boolean mask, same length as t
#     """
#     t = np.asarray(t, float)
#     iv = np.asarray(intervals, float)
#     if iv.size == 0:
#         return np.zeros_like(t, dtype=bool)

#     # sorted; overlaps allowed
#     iv = iv[np.argsort(iv[:, 0])]

#     mask = np.zeros(len(t), dtype=bool)
#     for start, stop in iv:
#         mask |= (t >= start) & (t < stop)   # [start, stop)
#     return mask

# # iv = nrem_epochs_with_barrage[nrem_epochs_with_barrage.durations>120].data
# # mask = mask_times_in_intervals(ca1_complete_df["center_s"].to_numpy(float), iv)

# # ca1_nrem_barrage = ca1_complete_df.loc[mask].copy()

# # centers_df = ca1_nrem_barrage
# out_df = isi_entropy_windows_with_tiles(
#     spike_times=spike_times,
#     centers_df=ca1_complete_df,
#     ripple_epochs=ripple_epochs,       # pass the objects; the function will read .data if present
#     barrage_epochs=None,
#     state_filter="NREM",
#     window=30.0,                       # ±15 s
#     min_spikes=3,                      # need ≥3 spikes in window
#     n_bins_isi=40
# )



In [388]:
# import pickle
# save_dir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats"
# os.makedirs(save_dir, exist_ok=True)

# # build filename and save
# fname = f"{animal_name}_{day}.pkl"
# save_path = os.path.join(save_dir, fname)

# with open(save_path, "wb") as f:
#     pickle.dump({"per_center": out_df,
#                  "animal_name": animal_name,
#                  "day": day}, f)

# print(f"✅ Saved {save_path}")
